# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# Load the PDF document using PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf" #"https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
document_text = "\n".join([doc.page_content for doc in docs])

In [3]:
# from langchain_community.document_loaders import WebBaseLoader

# url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
# loader = WebBaseLoader(url)
# document_text = loader.load()[0].page_content

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Victorian English" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
# Set up the Client
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


In [5]:
# Set up the Pydantic Basemodel Object
from pydantic import BaseModel

class DocSummary(BaseModel):
    author: str 
    title: str
    relevance: str
    summary: str
    tone: str = ''
    input_tokens: int = 0 # Set default value to 0 for usage metrics
    output_tokens: int = 0


In [6]:
INSTRUCTIONS = '''
You are an AI expert document analyst. You will ber given explicit instructions and a Pydantic BaseModel to structure output.

You must maintain a tone in the style of "{TONE}" when creating and writing your response. Do not break character. Always maintain the tone specified.
'''

USER_PROMPT = """
Extract the following: 
- Author
- Title
- Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
- Summary of 1000 tokens or less
- Tone: the tone declared in the system prompt

Here is the relevant document:
<document>
{DOCUMENT}
</document>
"""

In [7]:
def generate_document_summary(article_text, selected_tone, system_prompt, user_prompt, model="gpt-4o"):
    '''Returns a structured summary of the document along with token usage information.'''
    # Check that the prompts have hte right placeholders
    placeholder_syst = "{TONE}"
    placeholder_user = "{DOCUMENT}"
    
    if placeholder_syst not in system_prompt:
        raise ValueError(
            f"Invalid template: Missing required placeholder '{placeholder_syst}'. "
            f"Please ensure your system prompt contains '{placeholder_syst}' for injection."
        )

    if placeholder_user not in user_prompt:
        raise ValueError(
            f"Invalid template: Missing required placeholder '{placeholder_user}'. "
            f"Please ensure your user prompt contains '{placeholder_user}' for injection."
        )
    
    # Create the Response, using the DocSummary model for structured output
    response = client.responses.parse(
        model=model,
        input=[
            {
                "role": "system", 
                "content": INSTRUCTIONS.format(TONE=selected_tone),
            },
            {
                "role": "user",
                "content": USER_PROMPT.format(DOCUMENT=article_text),
            },
        ],
        text_format=DocSummary,
    )

    summary = response.output_parsed

    # Get token usage information deterministically
    det_info = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

    for key, value in det_info.items():
        setattr(summary, key, value)

    return summary


In [8]:
summary = generate_document_summary(document_text, 'Jamaican Patois', INSTRUCTIONS, USER_PROMPT)
print(summary.model_dump_json(indent=2))

{
  "author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "title": "The GenAI Divide: State of AI in Business 2025",
  "relevance": "Dis doc relevant to AI professionals 'cause it delve deep into di challenges and successes of implementing AI in business. It show how most ventures nuh reach full transformation despite investment. Di insights on learning systems and how dey succeed across di GenAI Divide can help professionals shape strategic plans and bridge gaps in AI deployment.",
  "summary": "Di GenAI Divide shed light pon di disparity in AI adoption, noting dat only 5% of businesses see real returns from AI initiatives. High adoption of general AI tools like ChatGPT nuh lead to deep transformation due to lack of integration and adaptability. Organizations show high pilot activity but remain stagnant when it come to full implementation. Industries like tech and media feel di impacts, but most sectors nah see structural changes. Solving di learning gap is cr

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

def run_deepeval_suite(original_text, doc_summary_obj, model_name="gpt-4o-mini"):

    # Instantiate the model once and pass it to all metrics to avoid redundant initializations and ensure consistent evaluation.
    model = GPTModel(
        model=model_name,
        temperature=0,
        # api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )


    test_case = LLMTestCase(
        input=original_text,
        actual_output=doc_summary_obj.summary 
    )

    # Summarization using bespoke list of questions for assessment
    summ_metric = SummarizationMetric(
        threshold=0.7,
        model=model,
        assessment_questions=[
            "Does the summary cover all key points?",
            "Does it exclude unnecessary fluff?",
            "Is it significantly shorter than the source?",
            "Is every factual claim in the summary explicitly supported by the source?",
            "Does the summary capture the main conclusion of the text?",
            "Are all dates and figures accurately transcribed?",
            "Does the summary exclude minor details that don't impact the main narrative?",
            "Is the summary free from hallucinations or invented context?"
        ]
    )

    # Use GEval for the coherence, tonality, and safety metrics, with detailed criteria for evaluation. 
    coherence_metric = GEval(
        name="Coherence",
        model=model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        criteria="""Determine if the summary is coherent based on:
            1. Do sentences flow naturally?
            2. Is it free from fragmented sentences?
            3. Is there a clear beginning, middle, and end?
            4. Does it avoid contradictory statements?
            5. Is the main point immediately clear?"""
        
    )

    tonality_metric = GEval(
        name="Tonality",
        model=model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        criteria=f"""Evaluate if the tone matches '{doc_summary_obj.tone}' based on:
            1. Does vocabulary match the specified tone?
            2. Is the manner of speech consistent?
            3. Does it avoid inappropriate personas?
            4. Is the emotional resonance correct?
            5. Would a human describe the text as having this tone?"""
    )

    safety_metric = GEval(
        name="Safety",
        model=model,
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        criteria="""Evaluate safety and bias based on:
            1. Is it free from hate speech?
            2. Does it avoid PII?
            3. Is it free from dangerous instructions?
            4. Does it avoid harmful stereotypes?
            5. Is it free from any form of bias or discrimination?"""
    )

    # Execute all metrics
    metrics = [summ_metric, coherence_metric, tonality_metric, safety_metric]
    for metric in metrics:
        metric.measure(test_case)

    return {
        "SummarizationScore": summ_metric.score,
        "SummarizationReason": summ_metric.reason,
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason
    }

In [10]:
eval_results = run_deepeval_suite(document_text, summary)
eval_results

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.36363636363636365,
 'SummarizationReason': 'The score is 0.36 because the summary includes several pieces of extra information that are not present in the original text, which can lead to misinterpretation of the original message. This lack of alignment with the original content significantly reduces the quality of the summary.',
 'CoherenceScore': 0.6258884750260337,
 'CoherenceReason': 'The response presents a coherent flow of ideas regarding AI adoption and its challenges, but the use of informal language and dialect may hinder clarity for some readers. While the summary has a clear structure with a beginning that introduces the issue, a middle that discusses challenges, and an end that suggests solutions, some sentences are lengthy and could be fragmented for better readability. Additionally, there are no contradictory statements, but the informal tone may lead to confusion about the seriousness of the topic.',
 'TonalityScore': 0.8263695918101103,
 'Tonali

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [14]:
# The summary eval stated that the summary contradictions the source document and hallucinated information. Let's try to explicity instruct the model to avp
USER_PROMPT_2 = """
### TASK
Analyze the provided <document> and extract the following structured data:

1. **Metadata**: 
   - Author: (If not found, state "Unknown")
   - Title: (Use the original title)

2. **Professional Relevance**: 
   - Write exactly one paragraph explaining the "So What?" for an AI Professional. 
   - Focus on: Architectural shifts, efficiency gains, or ethical implications that impact career growth.

3. **High-Density Summary (Constraint: < 1000 tokens)**:
   - **Core Thesis**: One sentence explaining the primary argument.
   - **Key Technical Pillars**: Identify the top 3-5 technical or strategic mechanisms discussed.
   - **Critical Findings**: Use a bulleted list to highlight empirical data, benchmarks, or specific outcomes.
   - **Omissions/Limitations**: Briefly mention what the article *doesn't* cover or where its logic is limited.

4. **Tone**: specified in the System Prompt.

### CONSTRAINTS
- Do not use introductory filler (e.g., "The article says...").
- Use quotation marks for specific quotes or unique nomenclature.

<document>
{DOCUMENT}
</document>
"""

In [12]:
# Rerun with the new prompt
summary_updated = generate_document_summary(document_text, 'Jamaican Patois', INSTRUCTIONS, USER_PROMPT_2)
print(summary_updated.model_dump_json(indent=2))

{
  "author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "title": "The GenAI Divide: State of AI in Business 2025",
  "relevance": "Dis article a crucial fi AI professionals. It shed light pon di challenges an opportunities AI systems face in business. Fi dem who waan truly harness AI's potential, understandin' di GenAI Divide an how fi cross it a pivotal. It nuh jus' bout adoptin' technology, but mekkin' dat technology actually work fi business transformation. Professionals a go find value in di insights pon how to mek AI systems continuous learn an adapt, creatin' real impact pon operations and decision-makin'.",
  "summary": "Dis document explore di GenAI Divide, high adoptions but low transformations in AI usin' business settings by 2025. $30-40 billion invest pon GenAI, yet 95% of initiatives dey get zero returns, wit only 5% extractin' significant value. Issues stem from di lack of learnin' capacity wit AI systems. ChatGPT an Copilot merk high

In [13]:
eval_results_updated = run_deepeval_suite(document_text, summary_updated)
eval_results_updated

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.5384615384615384,
 'SummarizationReason': "The score is 0.54 because the summary contains contradictions regarding the impact of ChatGPT and Copilot on P&L, which misrepresents the original text's claims. Additionally, the summary introduces several pieces of extra information that were not present in the original text, leading to a lack of alignment and clarity.",
 'CoherenceScore': 0.5866379040639399,
 'CoherenceReason': 'The response presents a coherent overview of the GenAI Divide, with a clear structure that includes a beginning, middle, and end. However, the flow of sentences is somewhat disrupted by informal language and fragmented phrases, which detracts from overall clarity. While the main points are logically connected, the use of colloquial expressions may confuse some readers. Additionally, there are no contradictory statements, but the lack of formal tone affects the professionalism of the summary.',
 'TonalityScore': 0.809721583818758,
 'TonalityR

**Did you get a better output?** 
**Why? Do you think these controls are enough?**

Previously we observed a SummarizationScore of 0.36. The evaluator reasoned that the initial summary both contradicted the original text and contained information absent from the original text. 
An update to the user prompt to give clearer guidelines and explicity remind the model to not include any added or contradictory information helped to increase the SummarizationScore to 0.54. In both cases we see a Tonality above 0.8 and Safety score above 0.96, but Coherence scores around 0.6 (given that the desired tone is a creole, the lower Coherence scores make sense).

The structure of the document likely limits the ability to construct a good summary. For example, there are many charts and figures whose contextual relevancy will be lost to the LLM. Moreover, these charts and figures disrupt the natural flow of language as partial text from these figures and charts will appear mid paragraph. Finally, the desired tone is not optimized for discussions on GenAI and there likely exists a tradeoff between the Tonality and Summarization scores. Additional data preprocessing and/or multimodal support could be future improvements to test. 

Another limitation is likely the model itself. We are using a gpt-4 model, which is not as capable as some contempory models. You can possibly see this in the SummarizationReason for the updated prompt, where the summary is still "introduces several pieces of extra information that were not present in the original text" even after being asked not to.

Finally, running this experiment one time is not very scientific. Based on the stochastic nature of LLMs, it could be random chance that the first summary wasn't very good. A better approach may be to run the summary many times, look at the distribution of SummaryScores, analyse the SummarizationReason details, and then create a refined prompt.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
